In [1]:
import sys
import os

# Añadir la carpeta 'src' al path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src')))

from models.model_handler import load_model

In [2]:
model = load_model("model.pth")

In [3]:
import pickle
import numpy as np
import torch
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder

with open('../../data/generated/all_types_200p_70000t_rp.pkl', 'rb') as file:
    data = pickle.load(file)

data = data.sample(frac=1).reset_index(drop=True)
coords_list = [np.stack([row[0], row[1]], axis=1) for row in data['coords']]
X = np.array(coords_list)  # Forma (n_samples, seq_length, 2)
X_tensor = torch.tensor(X, dtype=torch.float32)
# 2. Convertir las etiquetas a numéricas
le = LabelEncoder()
labels = le.fit_transform(data['trajectory_type'])
labels_tensor  = torch.tensor(labels, dtype=torch.long)
y_tensor = F.one_hot(labels_tensor, num_classes=7).float()
X_tensor = X_tensor.permute(0, 2, 1)  # Nueva forma: (n, 2, m)

In [4]:
import torch.nn.functional as F

# Seleccionar el primer ejemplo de X_test y añadir una dimensión batch
first_example = X_tensor[0].unsqueeze(0)  # shape: (1, 2, 200)

# Pasar por el modelo
model.eval()
with torch.no_grad():
    output = model(first_example)
    predicted_class = torch.argmax(output, dim=1).item()

# Mostrar el resultado
print("Predicción (índice de clase):", predicted_class)
print("Clase predicha:", le.inverse_transform([predicted_class])[0])
# Mostrar la clase real
true_class = torch.argmax(y_tensor[0]).item()
print("Clase real:", le.inverse_transform([true_class])[0])


Predicción (índice de clase): 1
Clase predicha: Brownian Motion
Clase real: Brownian Motion
